<a href="https://colab.research.google.com/github/Sanyam-3/Sam-pro/blob/main/optimized_tcp_congestion_analysis_with_mawi_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# installing tshark and Python libraries
!apt-get update -y
!apt-get install -y tshark wireshark
!pip install scapy pandas scikit-learn matplotlib numpy

In [ ]:
!wget -c https://mawi.wide.ad.jp/mawi/samplepoint-F/2026/202603021400.pcap.gz  #downloading the data of Mawi group Date: March 2nd 2026
!ls -lh # display the dowloaded data

In [ ]:
!gzip -dc 202603021400.pcap.gz > 202603021400.pcap #decompress the files
!ls -lh

In [ ]:
!editcap -c 50000 202603021400.pcap small_trace.pcap #splitting large PCAP into smaller files
!ls

In [ ]:
import glob # load packets into scapy
from scapy.all import rdpcap

print("Searching for available split PCAP files...")

pcap_files = sorted(glob.glob("small_trace_*.pcap"))

print("Found files:")
for f in pcap_files[:10]:
    print(f)

packets = rdpcap(pcap_files[0])

print("\nLoaded file:", pcap_files[0])
print("Total packets:", len(packets))

In [ ]:
tcp_packets = [p for p in packets if p.haslayer("TCP")] #finding how many tcp and quic packets are there!

quic_packets = [
    p for p in packets
    if p.haslayer("UDP") and (p["UDP"].sport == 443 or p["UDP"].dport == 443)
]

print("TCP packets:", len(tcp_packets))
print("QUIC packets:", len(quic_packets))

In [ ]:
from collections import defaultdict
from scapy.layers.inet import TCP, UDP, IP

def canonical_flow(src, dst, sport, dport, proto):#make both flow as one like A to B and B to A
    # bidirectional normalization
    if (src, sport) < (dst, dport):
        return (src, dst, sport, dport, proto)
    else:
        return (dst, src, dport, sport, proto)

tcp_flows = defaultdict(list)
quic_flows = defaultdict(list)

for p in packets:
    try:
        if not p.haslayer(IP):
            continue

        src = p[IP].src
        dst = p[IP].dst

        # TCP Flow Extraction
        if p.haslayer(TCP):
            sport = p[TCP].sport
            dport = p[TCP].dport

            fid = canonical_flow(src, dst, sport, dport, "TCP")
            tcp_flows[fid].append(p)

        # QUIC Flow Extraction (UDP 443)

        elif p.haslayer(UDP):
            sport = p[UDP].sport
            dport = p[UDP].dport

            if sport == 443 or dport == 443:
                fid = canonical_flow(src, dst, sport, dport, "QUIC")
                quic_flows[fid].append(p)

    except Exception:
        pass

print("Total TCP flows :", len(tcp_flows))
print("Total QUIC flows:", len(quic_flows))

In [ ]:
import numpy as np

tcp_dataset = []

for flow, pkts in tcp_flows.items():
    if len(pkts) < 5:
        continue

    sizes = [len(p) for p in pkts]
    times = [float(p.time) for p in pkts]

    duration = times[-1] - times[0]
    if duration <= 0:
        continue

    interarrival = np.diff(times)

    throughput = sum(sizes) / duration
    pkt_rate = len(pkts) / duration

    avg_size = np.mean(sizes)
    std_size = np.std(sizes)

    jitter = np.std(interarrival) if len(interarrival) > 0 else 0
    iat_mean = np.mean(interarrival) if len(interarrival) > 0 else 0
    iat_std = np.std(interarrival) if len(interarrival) > 0 else 0

    # TCP-specific congestion signals
    retrans = sum(1 for p in pkts if p["TCP"].flags == "R") # retransmiteed packets
    dup_acks = sum(1 for p in pkts if p["TCP"].flags == "A") #acknowledges packets

    tcp_dataset.append([
        "TCP",
        len(pkts),
        duration,
        throughput,
        pkt_rate,
        avg_size,
        std_size,
        jitter,
        iat_mean,
        iat_std,
        retrans,
        dup_acks
    ])

In [ ]:
quic_dataset = []

for flow, pkts in quic_flows.items():
    if len(pkts) < 5:
        continue

    sizes = [len(p) for p in pkts]
    times = [float(p.time) for p in pkts]

    duration = times[-1] - times[0]
    if duration <= 0:
        continue

    interarrival = np.diff(times)

    throughput = sum(sizes) / duration
    pkt_rate = len(pkts) / duration

    avg_size = np.mean(sizes)
    std_size = np.std(sizes)

    jitter = np.std(interarrival) if len(interarrival) > 0 else 0
    iat_mean = np.mean(interarrival) if len(interarrival) > 0 else 0
    iat_std = np.std(interarrival) if len(interarrival) > 0 else 0

    # QUIC does not expose retransmissions directly
    retrans = 0
    dup_acks = 0

    quic_dataset.append([
        "QUIC",
        len(pkts),
        duration,
        throughput,
        pkt_rate,
        avg_size,
        std_size,
        jitter,
        iat_mean,
        iat_std,
        retrans,
        dup_acks
    ])

In [ ]:
import pandas as pd

columns = [
    "protocol",
    "packet_count",
    "duration",
    "throughput",
    "packet_rate",
    "avg_packet_size",
    "packet_size_std",
    "jitter",
    "interarrival_mean",
    "interarrival_std",
    "retransmissions",
    "dup_acks"
]

df = pd.DataFrame(tcp_dataset + quic_dataset, columns=columns)

print("Dataset shape:", df.shape)
df.head()

In [ ]:
df["label"] = np.where(
    (df["throughput"] < df["throughput"].median()) & # low speed
    (df["jitter"] > df["jitter"].median()), # high delay variation
    0,   # external congestion
    1    # self-induced congestion
)

print(df["label"].value_counts())

In [ ]:
X = df.drop(["label", "protocol"], axis=1)
y = df["label"]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.7, random_state=42
)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(max_depth=10) #max 10 levels
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

print("Decision Tree Accuracy:", accuracy_score(y_test, dt_pred))
print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))

print("\nClassification Report:\n")
print(classification_report(y_test, rf_pred))

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(rf, X, y, cv=5)
print("Cross-validation accuracy:", scores.mean())

In [ ]:
total_flows = len(df)

external = sum(df["label"] == 0)
self_induced = sum(df["label"] == 1)

print("External congestion flows:", external)
print("Self-induced congestion flows:", self_induced)

print("External %:", external / total_flows * 100)
print("Self-induced %:", self_induced / total_flows * 100)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 1. Throughput vs Jitter Scatter

plt.figure()
plt.scatter(df["throughput"], df["jitter"])
plt.xlabel("Throughput")
plt.ylabel("Jitter")
plt.title("Throughput vs Jitter (TCP + QUIC)")
plt.show()



# 2. Congestion Distribution Bar

plt.figure()
counts = df["label"].value_counts().sort_index()
plt.bar(counts.index.astype(str), counts.values)
plt.xlabel("Congestion Type (0 = External, 1 = Self)")
plt.ylabel("Number of Flows")
plt.title("Distribution of Congestion Types")
plt.show()


# 3. Feature Importance (RF)

importances = rf.feature_importances_
feat_importance = pd.Series(importances, index=X.columns).sort_values(ascending=False)

plt.figure()
feat_importance.plot(kind="bar")
plt.xlabel("Features")
plt.ylabel("Importance")
plt.title("Random Forest Feature Importance")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, roc_curve, auc, ConfusionMatrixDisplay


# 1. Confusion Matrix with %

cm = confusion_matrix(y_test, rf_pred)

# Convert to percentages
cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

fig, ax = plt.subplots()
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(ax=ax, cmap="Blues", values_format="d")

# Overlay percentages
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i,
                f"\n{cm_percent[i, j]:.1f}%",
                ha="center", va="center", color="red", fontsize=10)

plt.title("Confusion Matrix - Random Forest")
plt.show()


# 2. ROC Curve

rf_probs = rf.predict_proba(X_test)[:, 1]

fpr, tpr, _ = roc_curve(y_test, rf_probs)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC Curve (AUC = {roc_auc:.3f})")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, roc_curve, auc, ConfusionMatrixDisplay

# Class labels
labels = ["External Congestion (0)", "Self Congestion (1)"]


# 1. Confusion Matrix

cm = confusion_matrix(y_test, rf_pred)

# Percentage per actual class
cm_percent = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, ax = plt.subplots(figsize=(6,5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(ax=ax, cmap="Blues", values_format="d")

# Overlay percentages
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i,
                f"\n({cm_percent[i, j]:.1f}%)",
                ha="center", va="center",
                color="red", fontsize=11)

plt.title("Confusion Matrix: Self vs External Congestion")
plt.show()